In [41]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [42]:
#Data Upload
df = pd.read_csv("https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv")
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


### In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.

### Data preparation <br>
Check if the missing values are presented in the features. <br>
If there are missing values:
1. For caterogiral features, replace them with 'NA'
2. For numerical features, replace with with 0.0

Split the data into 3 parts: train/validation/test with 60%/20%/20% distribution. Use train_test_split function for that with random_state=1

In [43]:
df.isna().sum().sort_values(ascending=False)

annual_income               181
industry                    134
lead_source                 128
employment_status           100
location                     63
number_of_courses_viewed      0
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [44]:
# For caterogiral features, replace them with 'NA'
df[df.select_dtypes(include="object").columns] = df[df.select_dtypes(include="object").columns].fillna("NA")

# For numerical features, replace with with 0.0
df[df.select_dtypes(exclude="object").columns] = df[df.select_dtypes(exclude="object").columns].fillna("0.0")

In [45]:
#Confirm whether the missing values have been worked on
df.isna().sum().sort_values(ascending=False)

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [46]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train,df_val = train_test_split(df_full_train,test_size=0.25,random_state=42)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

df_train.reset_index(drop=True)
df_val.reset_index(drop=True)
df_test.reset_index(drop=True)

len(df_train),len(df_val),len(df_test)

(876, 293, 293)

### Question 1: ROC AUC feature importance
ROC AUC could also be used to evaluate feature importance of numerical variables. <br>

Let's do that

For each numerical variable, use it as score (aka prediction) and compute the AUC with the y variable as ground truth.<br>
Use the training dataset for that
If your AUC is < 0.5, invert this variable by putting "-" in front<br>

(e.g. -df_train['balance'])<br>

AUC can go below 0.5 if the variable is negatively correlated with the target variable. You can change the direction of the correlation by negating this variable - then negative correlation becomes positive.<br>

Which numerical variable (among the following 4) has the highest AUC?<br>

1. lead_score
2. number_of_courses_viewed
3. interaction_count
4. annual_income